# 02 - Correlation Review

Correlation structure of the Open-Meteo-backed feature set, centered on European AQI.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(str(ROOT / ".env"), override=False)

from src.utils.mongo_client import get_database

sns.set_theme(style="whitegrid")

db = get_database()
collection = db["aqi_features_rawalpindi"]
data = pd.DataFrame(list(collection.find()))

if "_id" in data.columns:
    data = data.drop(columns=["_id"])

data["timestamp"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
data = data.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

data.head()

In [ ]:
numeric = data.select_dtypes(include="number").copy()
target_corr = (
    numeric.corr(numeric_only=True)["european_aqi"]
    .drop(labels=["european_aqi"], errors="ignore")
    .dropna()
    .sort_values(key=lambda series: series.abs(), ascending=False)
)

display(target_corr.to_frame(name="correlation"))

In [ ]:
top_features = target_corr.head(12).index.tolist()
plot_columns = ["european_aqi"] + top_features
corr_matrix = numeric[plot_columns].corr(numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(corr_matrix, cmap="RdBu_r", center=0, ax=axes[0], square=True)
axes[0].set_title("Top Correlations with European AQI")

sorted_corr = target_corr.head(15).sort_values()
colors = ["#dc2626" if value < 0 else "#2563eb" for value in sorted_corr]
sorted_corr.plot(kind="barh", ax=axes[1], color=colors)
axes[1].set_title("Top 15 Absolute Correlations")
axes[1].set_xlabel("Correlation")

plt.tight_layout()